# Projection vs local CLAMP model - placenta

💡 **Environment:** `clamp-analyses`

First-trimester placenta/decidua scRNA-seq (E-MTAB-6701), trophoblast vs other.

Each dataset is analysed twice: through a small CLAMP model fit on its own RNA-seq, and by projecting it into the adopted canonical ARCHS4 CLAMPfull model (`output/98_final_models/clampfull/canonical/archs4/CLAMPfull_canonical.rds`).

## Paper-defined mechanisms and cell types

Vento-Tormo et al., *Nature* (2018), DOI: [10.1038/s41586-018-0698-6](https://doi.org/10.1038/s41586-018-0698-6), characterized the molecular and cellular organization of the early human maternal-fetal interface.

**Expected molecular mechanisms:**
- Trophoblast differentiation into EVT and syncytiotrophoblast
- EVT invasion / epithelial-mesenchymal transition / spiral-artery remodelling
- dNK-trophoblast HLA-C / HLA-E / HLA-G recognition
- dNK chemokine signaling / immune-cell recruitment
- Immune checkpoint / maternal-fetal immune tolerance
- Adenosine-mediated immunoregulation
- dNK1 glycolytic metabolic priming

**Expected cell types:**
- Trophoblast
- Extravillous trophoblast (EVT)
- Syncytiotrophoblast (SCT)
- Villous cytotrophoblast (VCT)
- Decidual natural killer cells (dNK1 / dNK2 / dNK3)
- Decidual stromal cells
- Decidual macrophages / maternal immune cells


In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(here)
  library(ggplot2)
})

source(here("scripts", "archs4", "projections", "plots.R"))

agg_dir   <- snakemake@params[['agg_dir']]
group_sel <- snakemake@params[['group_label']]
prod_root <- snakemake@params[['prod_root']]
cat('group:', group_sel, '| tables from:', agg_dir, '\n')


In [ ]:
placenta_dataset <- "placenta_EMTAB6701"

ora_dbs <- c("canonical", "hallmark", "cellmarker", "azimuth")
hits <- rbindlist(lapply(c("local", "ARCHS4"), function(mdl) rbindlist(lapply(ora_dbs, function(db) {
  f <- file.path(prod_root, placenta_dataset, "ora", mdl, db, "enrichment.csv.gz")
  if (!file.exists(f)) return(NULL)
  d <- fread(f)
  if (!nrow(d)) return(NULL)
  d[, `:=`(model = mdl, database = db)]
  d
}))))
setnames(hits, c("ID", "p.adjust"), c("term", "fdr"), skip_absent = TRUE)
hits <- hits[fdr < 0.05 & Count >= 3]

manual_mechanisms <- list(
  "EVT invasion / EMT" =
    list(pattern = "EPITHELIAL_MESENCHYMAL_TRANSITION",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "Spiral-artery remodelling" =
    list(pattern = "CELL_SURFACE_INTERACTIONS_AT_THE_VASCULAR_WALL",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "dNK-trophoblast HLA-C / HLA-E / HLA-G interactions" =
    list(pattern = "HLAC_ALLOTYPES_INTERACTIONS_WITH_KIR_ON_DNK_CELLS",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "dNK chemokine / immunomodulatory signaling" =
    list(pattern = "CHEMOKINE_SIGNALING",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "Immune checkpoint / maternal-fetal immune tolerance" =
    list(pattern = "CO_INHIBITION_BY_PD_1|IMMUNOREGULATORY_INTERACTIONS_BETWEEN_A_LYMPHOID_AND_A_NON_LYMPHOID_CELL",
         exclude_lv = c("LV1269", "LV948", "LV90"),
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "Adenosine-mediated immunoregulation" =
    list(pattern = "ADORA2B",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "dNK1 glycolytic metabolic priming" =
    list(pattern = "KEGG_MEDICUS_REFERENCE_GLYCOLYSIS",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "Trophoblast" =
    list(pattern = "trophoblast", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Extravillous trophoblast (EVT)" =
    list(pattern = "Extravillous Trophoblasts", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Syncytiotrophoblast (SCT)" =
    list(pattern = "Syncytiotrophoblasts And Villous Cytotrophoblasts", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Villous cytotrophoblast (VCT)" =
    list(pattern = "Syncytiotrophoblasts And Villous Cytotrophoblasts", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Decidual NK / dNK" =
    list(pattern = "UTERINE_NATURAL_KILLER", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Decidual stromal cells" =
    list(pattern = "Stromal", prefer_cellmarker = TRUE, exclude_lv = c("LV90", "LV126"),
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Decidual macrophages / maternal myeloid cells" =
    list(pattern = "Macrophage", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other")
)

find_best <- function(hm, spec) {
  d <- hm[grepl(spec$pattern, term, ignore.case = TRUE)]
  if (!is.null(spec$exclude_lv)) d <- d[!LV %chin% spec$exclude_lv]
  if (!nrow(d)) return(NULL)
  if (isTRUE(spec$prefer_cellmarker)) {
    d[, rnk := fifelse(database == "cellmarker", 0L, 1L)]
    setorder(d, rnk, fdr)
    d[, rnk := NULL]
  } else setorder(d, fdr)
  d[1]
}

manual_recovery <- rbindlist(lapply(c("local", "ARCHS4"), function(mdl) {
  hm <- hits[model == mdl]
  rbindlist(lapply(names(manual_mechanisms), function(nm) {
    spec <- manual_mechanisms[[nm]]
    r <- find_best(hm, spec)
    if (is.null(r)) {
      data.table(mechanism = nm, LV = NA_character_, term = NA_character_, fdr = NA_real_,
                 database = NA_character_, model = mdl, category = spec$category,
                 target_contrast = spec$target_contrast)
    } else {
      data.table(mechanism = nm, LV = r$LV, term = r$term, fdr = r$fdr,
                 database = r$database, model = mdl, category = spec$category,
                 target_contrast = spec$target_contrast)
    }
  }))
}))
manual_recovery[, LV_label := LV]

combined <- rbindlist(lapply(c("local", "ARCHS4"), function(mdl) {
  both <- manual_recovery[model == mdl & mechanism %chin% c("Extravillous trophoblast (EVT)", "Syncytiotrophoblast (SCT)") & !is.na(LV)]
  if (!nrow(both)) {
    return(data.table(mechanism = "Trophoblast differentiation into EVT / SCT", LV = NA_character_,
                       LV_label = NA_character_, term = NA_character_, fdr = NA_real_,
                       database = NA_character_, model = mdl, category = "molecular_mechanism",
                       target_contrast = "trophoblast_vs_other"))
  }
  setorder(both, fdr)
  primary <- both[1]
  data.table(mechanism = "Trophoblast differentiation into EVT / SCT", LV = primary$LV,
             LV_label = paste(unique(both$LV), collapse = " / "),
             term = paste(sort(unique(both$term)), collapse = " / "),
             fdr = primary$fdr, database = primary$database, model = mdl,
             category = "molecular_mechanism", target_contrast = "trophoblast_vs_other")
}))
manual_recovery <- rbind(manual_recovery, combined)

row_order <- c(
  "Trophoblast differentiation into EVT / SCT", "EVT invasion / EMT", "Spiral-artery remodelling",
  "dNK-trophoblast HLA-C / HLA-E / HLA-G interactions", "dNK chemokine / immunomodulatory signaling",
  "Immune checkpoint / maternal-fetal immune tolerance", "Adenosine-mediated immunoregulation",
  "dNK1 glycolytic metabolic priming", "Trophoblast", "Extravillous trophoblast (EVT)",
  "Syncytiotrophoblast (SCT)", "Villous cytotrophoblast (VCT)", "Decidual NK / dNK",
  "Decidual stromal cells", "Decidual macrophages / maternal myeloid cells")
manual_recovery[, recovered := !is.na(LV)]
manual_recovery[, mechanism := factor(mechanism, levels = row_order)]
setorder(manual_recovery, mechanism, model)

print(manual_recovery[, .(mechanism, model, LV_label, pathway = term, fdr, recovered)])


In [ ]:
source(here("scripts", "archs4", "common.R"))

read_gmt_sets <- function(path) {
  x <- strsplit(readLines(path, warn = FALSE), "\t", fixed = TRUE)
  out <- lapply(x, function(row) unique(row[-c(1L, 2L)]))
  names(out) <- vapply(x, `[[`, "", 1L)
  out
}
read_cellmarker_sets <- function(path, sheet, term_col, gene_col) {
  x <- data.table::as.data.table(readxl::read_excel(path, sheet = sheet))
  x <- x[!is.na(get(term_col)) & !is.na(get(gene_col)),
         .(term = as.character(get(term_col)), gene = as.character(get(gene_col)))]
  split(x$gene, x$term)
}
gene_sets <- list(
  canonical  = read_gmt_sets(here("data", "pathways", "c2.cp.v2026.1.Hs.symbols.gmt")),
  hallmark   = read_gmt_sets(here("data", "pathways", "h.all.v2026.1.Hs.symbols.gmt")),
  azimuth    = read_gmt_sets(here("data", "pathways", "Azimuth_2023.txt")),
  cellmarker = read_cellmarker_sets(here("data", "pathways", "Cell_marker_Human.xlsx"),
                                     "human", "cell_name", "Symbol")
)

z_registry <- fread(file.path(prod_root, placenta_dataset, "mechanism_models.tsv"))
z_mats <- list(
  ARCHS4 = read_matrix_csv(here("output", "98_final_models", "clampfull", "canonical", "archs4", "Z.csv")),
  local  = read_matrix_csv(here(z_registry[model == "local"]$z))
)

top_pct <- 0.01
recovered_rows <- manual_recovery[recovered == TRUE]

gene_loadings <- rbindlist(lapply(seq_len(nrow(recovered_rows)), function(i) {
  r <- recovered_rows[i]
  z <- z_mats[[r$model]]
  members <- intersect(unique(unlist(gene_sets[[r$database]][strsplit(r$term, " / ", fixed = TRUE)[[1]]])), rownames(z))
  n_top <- max(1L, ceiling(nrow(z) * top_pct))
  ord <- order(z[, r$LV], decreasing = TRUE)[seq_len(n_top)]
  data.table(dataset = placenta_dataset, comparison_id = as.character(r$mechanism), model = r$model,
             rank = seq_along(ord), gene = rownames(z)[ord], loading = z[ord, r$LV],
             is_gene_set = rownames(z)[ord] %chin% members, in_top_loading_set = TRUE,
             n_gene_set_in_universe = length(members))
}))

comparisons <- copy(manual_recovery)
setnames(comparisons, "term", "gene_set")
comparisons[, `:=`(dataset = placenta_dataset, comparison_id = as.character(mechanism), top_pct = top_pct)]
comparisons <- merge(
  comparisons,
  gene_loadings[, .(n_gene_set_in_top_loading = sum(is_gene_set),
                     n_gene_set_in_universe = max(n_gene_set_in_universe)),
                by = .(comparison_id, model)],
  by = c("comparison_id", "model"), all.x = TRUE)
comparisons[is.na(n_gene_set_in_top_loading), n_gene_set_in_top_loading := 0L]
comparisons[is.na(n_gene_set_in_universe), n_gene_set_in_universe := 0L]

comparison_tests <- comparisons[, {
  a <- .SD[model == "ARCHS4"]
  b <- .SD[model == "local"]
  p <- NA_real_
  arch_fraction <- NA_real_
  local_fraction <- NA_real_
  if (nrow(a) == 1L && nrow(b) == 1L && isTRUE(a$recovered) && isTRUE(b$recovered) &&
      identical(a$gene_set, b$gene_set) &&
      a$n_gene_set_in_universe > 0L && b$n_gene_set_in_universe > 0L) {
    arch_fraction <- a$n_gene_set_in_top_loading / a$n_gene_set_in_universe
    local_fraction <- b$n_gene_set_in_top_loading / b$n_gene_set_in_universe
    p <- fisher.test(matrix(c(a$n_gene_set_in_top_loading,
                              a$n_gene_set_in_universe - a$n_gene_set_in_top_loading,
                              b$n_gene_set_in_top_loading,
                              b$n_gene_set_in_universe - b$n_gene_set_in_top_loading),
                            nrow = 2, byrow = TRUE), alternative = "greater")$p.value
  }
  .(p_value = p, arch_fraction = arch_fraction, local_fraction = local_fraction)
}, by = comparison_id]
comparison_tests[, p_adj := p.adjust(p_value, method = "BH")]
comparisons <- merge(comparisons, comparison_tests, by = "comparison_id", all.x = TRUE)

comparisons[, comparison_note := ""]
comparisons[model == "local" & !recovered, comparison_note := "Not recovered"]
comparisons[recovered == TRUE & is.na(p_adj), comparison_note := "Different matched pathway"]
comparisons[recovered == TRUE & !is.na(p_adj), comparison_note := paste0(
  fifelse(arch_fraction > local_fraction, "ARCHS4 > Placenta model",
          fifelse(arch_fraction < local_fraction, "Placenta model > ARCHS4", "equal fraction")),
  " (FDR ", formatC(p_adj, format = "e", digits = 1), ")")]


In [ ]:
notebook_mechanism_heatmap <- function(comparisons, dsel, mechanism_order) {
  d <- copy(comparisons[dataset == dsel])
  if (!nrow(d)) return(NULL)
  d[, lab := fifelse(recovered,
                     sprintf("%s\n%d/%d set genes", LV, n_gene_set_in_top_loading, n_gene_set_in_universe),
                     "Not recovered")]
  d[, gene_set_fraction_in_top_loading := fifelse(recovered & n_gene_set_in_universe > 0,
                                                  n_gene_set_in_top_loading / n_gene_set_in_universe, 0)]
  d[, mechanism := as.character(mechanism)]
  d[, mechanism_label := fifelse(category == "cell_type", paste0("cell type: ", mechanism), mechanism)]
  label_order <- vapply(mechanism_order, function(nm) {
    cat_i <- unique(d[mechanism == nm]$category)
    if (length(cat_i) && cat_i[1] == "cell_type") paste0("cell type: ", nm) else nm
  }, character(1))
  d[, mechanism_label := factor(mechanism_label, levels = rev(label_order))]
  d[, model_label := factor(model, levels = c("local", "ARCHS4"),
                            labels = c(proj_local_model_label(dsel), "ARCHS4"))]
  ggplot(d, aes(model_label, mechanism_label, fill = gene_set_fraction_in_top_loading)) +
    geom_tile(colour = "white", linewidth = 0.6) +
    geom_text(aes(label = lab), size = 2.5, lineheight = 0.95,
              colour = ifelse(d$gene_set_fraction_in_top_loading > 0.6, "white", "grey15")) +
    scale_fill_gradient(low = "#f7f7f7", high = "#332288", limits = c(0, 1), name = "gene-set\nfraction") +
    guides(fill = guide_colourbar(barheight = grid::unit(34, "pt"), barwidth = grid::unit(7, "pt"))) +
    labs(x = NULL, y = NULL, title = "Mechanism recovery: gene-set fraction in top 1% loading genes") +
    proj_theme() +
    theme(legend.title = element_text(size = 8.5, lineheight = 0.95), legend.text = element_text(size = 8))
}

options(repr.plot.width = 8, repr.plot.height = 2.5 + 0.5 * uniqueN(comparisons$mechanism))
print(notebook_mechanism_heatmap(comparisons, placenta_dataset, row_order))


In [ ]:
lv_stats <- fread(file.path(agg_dir, 'lv_stats_long.csv'))[dataset == placenta_dataset]

heatmap_input <- copy(manual_recovery)
setnames(heatmap_input, "term", "gene_set")
heatmap_input[, `:=`(dataset = placenta_dataset, comparison_id = as.character(mechanism),
                     mechanism = as.character(mechanism), ora_fdr = fdr)]

n_rows <- uniqueN(heatmap_input[recovered == TRUE, .(mechanism, LV)])
options(repr.plot.width = 12, repr.plot.height = 2.5 + 0.5 * n_rows)
print(proj_expected_mechanism_heatmap(lv_stats, heatmap_input, placenta_dataset))


In [ ]:
notebook_gene_set_pair_plot <- function(comparisons, loadings, dsel, comparison_key) {
  s <- comparisons[dataset == dsel & comparison_id == comparison_key]
  if (!nrow(s)) return(NULL)
  s[, model := factor(model, levels = c("ARCHS4", "local"))]
  d <- loadings[dataset == dsel & comparison_id == comparison_key]
  d <- d[in_top_loading_set == TRUE]
  d[, model := factor(model, levels = c("ARCHS4", "local"))]
  if (!nrow(d[model == "ARCHS4"])) return(NULL)

  labels <- d[is_gene_set == TRUE][order(model, rank), head(.SD, 5L), by = model]
  s[, label := fifelse(recovered,
                        sprintf("%d/%d in top %.0f%%", n_gene_set_in_top_loading,
                                n_gene_set_in_universe, 100 * top_pct),
                        "Not recovered")]
  s[nzchar(comparison_note), label := paste(label, comparison_note, sep = "\n")]
  missing <- s[recovered == FALSE]
  model_titles <- c(ARCHS4 = "ARCHS4 projection", local = proj_local_model_label(dsel))
  strip_labels <- setNames(unname(model_titles[as.character(s$model)]), as.character(s$model))
  plot_title <- if (any(s$recovered)) {
    paste(vapply(strsplit(s[recovered == TRUE]$gene_set[1], " / ", fixed = TRUE)[[1]],
                 proj_pretty_term, character(1)), collapse = " / ")
  } else as.character(s$mechanism[1])
  plot_title <- paste(strwrap(plot_title, width = 90), collapse = "\n")

  ggplot(d, aes(rank, loading)) +
    geom_point(data = d[is_gene_set == FALSE], colour = "grey78", size = 0.18,
               alpha = 0.65) +
    geom_point(data = d[is_gene_set == TRUE], aes(colour = model), size = 0.75,
               alpha = 0.95) +
    ggrepel::geom_text_repel(data = labels, aes(label = gene, colour = model),
                            size = 2.4, fontface = "italic", seed = 42,
                            max.overlaps = Inf, min.segment.length = 0,
                            segment.size = 0.2, segment.colour = "grey55",
                            box.padding = 0.15, point.padding = 0.08,
                            show.legend = FALSE) +
    geom_text(data = s[recovered == TRUE], aes(x = Inf, y = Inf, label = label),
              inherit.aes = FALSE, hjust = 1.05, vjust = 1.15, size = 2.7,
              colour = "grey20") +
    geom_text(data = missing, aes(x = Inf, y = Inf, label = label),
              inherit.aes = FALSE, hjust = 1.05, vjust = 1.15, size = 3.2,
              colour = "grey30") +
    facet_wrap(~model, nrow = 1, scales = "free_y", drop = FALSE,
               labeller = labeller(model = as_labeller(strip_labels))) +
    scale_colour_manual(values = PROJ_MODEL_COLOURS) +
    scale_x_continuous(expand = expansion(mult = c(0.015, 0.08))) +
    labs(x = "Top 1% genes ranked by descending loading", y = "Gene loading", title = plot_title) +
    theme_classic(base_size = 9) +
    theme(strip.background = element_blank(),
          strip.text = element_text(face = "bold"),
          plot.title = element_text(size = 11, hjust = 0.5),
          panel.grid = element_blank(),
          legend.position = "none")
}

for (nm in row_order) {
  p <- notebook_gene_set_pair_plot(comparisons, gene_loadings, placenta_dataset, nm)
  if (!is.null(p)) {
    options(repr.plot.width = 8, repr.plot.height = 3.2)
    print(p)
  }
}
